In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

sns.set_theme(style="whitegrid")

In [ ]:
movies = pd.read_csv("../data/processed/movies_cleaned.csv")

# Convert date column back to datetime
movies["released_date"] = pd.to_datetime(
    movies["released_date"],
    errors="coerce"
)

print("Dataset shape:", movies.shape)

In [ ]:
movies.head()

In [ ]:
movies.info()

In [ ]:
movies.describe().T

In [ ]:
# Missing-value overview 
missing = pd.DataFrame({
    "missing_count": movies.isna().sum(),
    "missing_percentage": (
        movies.isna().mean() * 100
    ).round(2)
})

missing.sort_values(
    "missing_percentage",
    ascending=False
)

In [ ]:
# Movies by Year 
movies_per_year = (
    movies["year"]
    .value_counts()
    .sort_index()
)

movies_per_year

In [ ]:
# Visualization
plt.figure(figsize=(12, 6))

sns.barplot(
    x=movies_per_year.index,
    y=movies_per_year.values
)

plt.title("Number of Movies in the Dataset by Year")
plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Average Score Over Time
average_score_by_year = (
    movies.groupby("year")["score"]
    .mean()
    .sort_index()
)

average_score_by_year

In [ ]:
# Visualization
plt.figure(figsize=(12, 6))

sns.lineplot(
    x=average_score_by_year.index,
    y=average_score_by_year.values,
    marker="o"
)

plt.title("Average Movie Score by Year")
plt.xlabel("Year")
plt.ylabel("Average Score")

plt.tight_layout()
plt.show()

In [ ]:
# Genre Analysis 
genre_counts = movies["genre"].value_counts()

genre_counts

In [ ]:
# Visualization 
plt.figure(figsize=(12, 7))

sns.barplot(
    x=genre_counts.values,
    y=genre_counts.index
)

plt.title("Movie Distribution by Genre")
plt.xlabel("Number of Movies")
plt.ylabel("Genre")

plt.tight_layout()
plt.show()

In [ ]:
# Average Score by Genre 
genre_score = (
    movies.groupby("genre")["score"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

genre_score

In [ ]:
# Visualization 
plt.figure(figsize=(12, 7))

sns.barplot(
    data=genre_score.reset_index(),
    x="mean",
    y="genre"
)

plt.title("Average Movie Score by Genre")
plt.xlabel("Average Score")
plt.ylabel("Genre")

plt.tight_layout()
plt.show()

In [ ]:
movies["runtime"].describe()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    movies["runtime"].dropna(),
    bins=30,
    kde=True
)

plt.title("Distribution of Movie Runtime")
plt.xlabel("Runtime (minutes)")
plt.ylabel("Number of Movies")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=movies,
    x="runtime",
    y="score",
    alpha=0.5
)

plt.title("Movie Runtime vs Score")
plt.xlabel("Runtime (minutes)")
plt.ylabel("Score")

plt.tight_layout()
plt.show()

In [ ]:
#Identifying long movies for 
long_movies = movies[movies["runtime"] > 180][
    ["name", "runtime", "year", "genre", "score"]
].sort_values("runtime", ascending=False)

long_movies

In [ ]:
# Collecting movies with the same runtime range 
runtime_bins = pd.cut(
    movies["runtime"],
    bins=[0, 90, 120, 150, 180, 240, 400],
    labels=[
        "Under 90",
        "90–120",
        "120–150",
        "150–180",
        "180–240",
        "240+"
    ]
)

runtime_distribution = runtime_bins.value_counts().sort_index()

runtime_distribution

In [ ]:
runtime_distribution.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Movies by Runtime Range")
plt.xlabel("Runtime Range (minutes)")
plt.ylabel("Number of Movies")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Checking the relationship between runtime and movie score 
runtime_score_corr = movies["runtime"].corr(movies["score"])

print("Correlation between runtime and score:", runtime_score_corr)
runtime_score = movies.groupby(runtime_bins)["score"].agg(
    ["mean", "count"]
)

runtime_score

In [ ]:
# Let's visualized it 
plt.figure(figsize=(10, 6))

sns.barplot(
    x=runtime_score.index,
    y=runtime_score["mean"]
)

plt.title("Average Movie Score by Runtime Range")
plt.xlabel("Runtime Range")
plt.ylabel("Average Score")

plt.tight_layout()
plt.show()

In [ ]:
budget_data = movies[movies["budget"].notna()].copy()

print("Movies with known budget:", len(budget_data))